# Introductie tot Cross Validation

In deze les zullen we een veel diepgaandere duik nemen in verschillende methoden van cross validation. Ook zullen we discussiëren over de algemene filosofie achter cross validation. Een goede officiële documentatiegids is hier te vinden: https://scikit-learn.org/stable/modules/cross_validation.html

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data voorbeeld

In [2]:
df = pd.read_csv("Advertising.csv")

In [3]:
df.head()

,TV,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


----
## Reminder: Train | Test Split Procedure 

0. Clean en pas data aan waar nodig voor X en y
1. Verdeel de data in Train/Test voor zowel X als y
2. Pas/Train Scaler toe op de Trainingsdata van X
3. Schaal X Testdata
4. Maak Model
5. Pas/Train Model toe op X Trainingsdata
6. Evalueer Model op X Testdata (door voorspellingen te maken en te vergelijken met Y_test)
7. Pas parameters aan indien nodig en herhaal stappen 5 en 6.

In [4]:
## Maak X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

**Maak het Model**

In [6]:
from sklearn.linear_model import Ridge

In [8]:
# Wat vinden jullie van de keuze voor alpha?
model = Ridge(alpha=100)

In [9]:
model.fit(X_train,y_train)

Ridge(alpha=100)

In [10]:
y_pred = model.predict(X_test)

**Evaluatie**

In [11]:
from sklearn.metrics import mean_squared_error

In [12]:
mean_squared_error(y_test,y_pred)

7.341775789034128

**Pas alpha aan en evalueer opnieuw**

In [13]:
model = Ridge(alpha=1)

In [14]:
model.fit(X_train,y_train)

Ridge(alpha=1)

In [15]:
y_pred = model.predict(X_test)

**Nieuwe Evaluatie**

In [16]:
mean_squared_error(y_test,y_pred)

2.3190215794287514

Veel beter! We kunnen dit herhalen tot we tevreden zijn met de prestatie-indicatoren. (We hebben eerder aangetoond dat RidgeCV dit voor ons kan doen, maar het doel van deze les is om het CV-proces te generaliseren voor elk model).

Voordeel van deze werkwijze: eenvoudig te begrijpen.  We doen één enkele split dus ook weinig rekenkracht nodig.  
Nadeel: hoe de optimale waarde voor hyperparameter vinden?  Traag en herhalend proces... We zouden een for-loop kunnen implementere, waardoor we in de richting van grid search gaan (zie verder).  Belangrijkste nadeel?  De MSE die we hier berekenden is niet de meest faire evaluatie.  Waarom? Omdat we hier bij het manuele proces onze alpha hebben aangepast op basis van de resultaten in het eerste model! De hyperparameter is dus bijgestuurd op basis van de test set...  Daarom: beetje data opzij houden voor een faire evaluatie.

### Terug naar de theorie!

----
## Train | Validation | Test Split Procedure 

Dit wordt vaak ook een "hold-out" set genoemd, omdat je parameters niet zou moeten aanpassen op basis van de uiteindelijke testset, maar het in plaats daarvan *alleen* moet gebruiken om de verwachte uiteindelijke prestaties te rapporteren.

1. Clean de data en pas deze zo nodig aan voor X en y
2. Verdeel de data in Trainings-/Validatie-/Testset voor zowel X als y
3. Fit/Train Scaler op de trainingsdata van X
4. Scale X Evaluation data
5. Creëer Model
6. Fit/Train Model op X Training data
7. Evalueer Model op X Evaluation data (door voorspellingen te maken en deze te vergelijken met Y_eval)
8. Pas indien nodig parameters aan en herhaal stappen 5 en 6
9. Verkrijg de uiteindelijke metingen op de Testset (na dit mag je niet meer terug om aan te passen!)


In [17]:
## Maak X and y
X = df.drop('sales',axis=1)
y = df['sales']

In [18]:
####################################################################
#### SPLIT TWEE KEER! Hier maken we TRAIN | VALIDATION | TEST  #####
####################################################################
from sklearn.model_selection import train_test_split

# 70% van de data is training data, zet 30% opzij
X_train, X_OTHER, y_train, y_OTHER = train_test_split(X, y, test_size=0.3, random_state=101)

# Resterende 30% splits je in evaluation en test set
# Elk is 15% van de originele data - OPGEPAST: de volgorde is belangrijk!
X_eval, X_test, y_eval, y_test = train_test_split(X_OTHER, y_OTHER, test_size=0.5, random_state=101)

In [19]:
# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_eval = scaler.transform(X_eval)
X_test = scaler.transform(X_test)

**Maak Model**

In [20]:
from sklearn.linear_model import Ridge

In [22]:
# PBewust slechte keuze voor alpha!
model = Ridge(alpha=100)

In [23]:
model.fit(X_train,y_train)

Ridge(alpha=100)

In [24]:
y_eval_pred = model.predict(X_eval)

**Evaluatie**

In [25]:
from sklearn.metrics import mean_squared_error

In [26]:
mean_squared_error(y_eval,y_eval_pred)

7.320101458823871

*i Waarom is dit niet identiek aan de waarde die we hierboven kregen? i*

**Pas Parameters aan fit evalueer opnieuw**

In [27]:
model = Ridge(alpha=1)

In [28]:
model.fit(X_train,y_train)

Ridge(alpha=1)

In [29]:
y_eval_pred = model.predict(X_eval)

**Nieuwe Evaluatie**

In [30]:
mean_squared_error(y_eval,y_eval_pred)

2.383783075056986

**Finale Evaluatie (Na dit: geen aanpassingen meer hierna!)**

In [31]:
y_final_test_pred = model.predict(X_test)

In [32]:
mean_squared_error(y_test,y_final_test_pred)

2.2542600838005176

Deze onderstaande MSE is wat we rapporteren.  Het model wat we finaal deployen is een model dat getraind is op de volledige dataset!

### Terug naar de theorie!

----
## Cross Validation met cross_val_score

----

<img src="grid_search_cross_validation.png">

----

In [33]:
## MAAK X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [34]:
model = Ridge(alpha=100)  # slecht model; alpha veel te hoog gekozen)

In [35]:
from sklearn.model_selection import cross_val_score

In [36]:
# SCORING OPTIONS:
# https://scikit-learn.org/stable/modules/model_evaluation.html
# je kan elk model kiezen als estimator!
scores = cross_val_score(model,X_train,y_train,
                         scoring='neg_mean_squared_error',cv=5)

In [37]:
scores # hier zie je de resultaten PER FOLD.  Sommige zijn goed, andere geven slechte resultaten!

array([ -9.32552967,  -4.9449624 , -11.39665242,  -7.0242106 ,
        -8.38562723])

In [50]:
# Gemiddelde van de MSE scores (terug positief maken)
abs(scores.mean())

8.215396464543607

In [43]:
# Dit is iets slechter dan voorheen omdat het een gemiddelde over de folds is, en in dit geval zijn er 
# enkele folds die slecht presteren en dus het gemiddelde slechter maken.  Dit is vaak zo.

**Pas model aan volgens metrics**

In [38]:
model = Ridge(alpha=1)

In [39]:
# SCORING OPTIONS:
# https://scikit-learn.org/stable/modules/model_evaluation.html
scores = cross_val_score(model,X_train,y_train,
                         scoring='neg_mean_squared_error',cv=5)

In [40]:
# Gemiddelde van de MSE scores (terug positief maken)
abs(scores.mean())

3.3448392965306946

**Finale Evaluatie (Niet meer aanpaseen na dit!)**

In [41]:
# Eerst een "gewone" fit op de data
model.fit(X_train,y_train)

Ridge(alpha=1)

In [42]:
y_final_test_pred = model.predict(X_test)

In [43]:
mean_squared_error(y_test,y_final_test_pred)

2.3190215794287514

### Terug naar de theorie!

----
# Cross Validation met cross_validate

De cross_validate functie verschilt van cross_val_score op twee manieren:

cross_validate staat toe meerdere evaluatiemetrics te specificeren.
Het geeft een dict terug met fit-tijden, score-tijden (en optioneel ook trainingsscores en gefitte schatters) naast de testscore.
Voor een enkele metric evaluatie, waar de scoring parameter een string, callable of None is, zullen de keys de volgende zijn:

    - ['test_score', 'fit_time', 'score_time']
    
En voor meerdere metric evaluatie, is de teruggegeven waarde een dict met de volgende keys:

    ['test_<scorer1_naam>', 'test_<scorer2_naam>', 'test_<scorer...>', 'fit_time', 'score_time']

return_train_score staat standaard op False om rekentijd te besparen. Om de scores ook op de trainingset te evalueren moet dit op True gezet worden.

In [44]:
## CREATE X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [45]:
model = Ridge(alpha=100)

In [46]:
from sklearn.model_selection import cross_validate

In [47]:
# SCORING OPTIONS:
# https://scikit-learn.org/stable/modules/model_evaluation.html
scores = cross_validate(model,X_train,y_train,
                         scoring=['neg_mean_absolute_error'],cv=5)

In [48]:
scores  # één enkele metric doorgegeven

{'fit_time': array([0.00274992, 0.00163102, 0.00044107, 0.00049114, 0.00050592]),
 'score_time': array([0.00067425, 0.00039816, 0.00019693, 0.0002439 , 0.00023556]),
 'test_neg_mean_absolute_error': array([-2.31243044, -1.74653361, -2.56211701, -2.01873159, -2.27951906])}

In [50]:
# SCORING OPTIONS:
# https://scikit-learn.org/stable/modules/model_evaluation.html
scores = cross_validate(model,X_train,y_train,
                         scoring=['neg_mean_absolute_error','neg_mean_squared_error','max_error'],cv=5)

In [51]:
scores  # resultaat van elke fold op meerdere metrics

{'fit_time': array([0.00229812, 0.00079584, 0.000453  , 0.00043941, 0.00046253]),
 'score_time': array([0.00087452, 0.00045705, 0.00041699, 0.00041986, 0.00039983]),
 'test_neg_mean_absolute_error': array([-2.31243044, -1.74653361, -2.56211701, -2.01873159, -2.27951906]),
 'test_neg_mean_squared_error': array([ -9.32552967,  -4.9449624 , -11.39665242,  -7.0242106 ,
         -8.38562723]),
 'test_max_error': array([ -6.44988486,  -5.58926073, -10.33914027,  -6.61950405,
         -7.75578515])}

In [52]:
pd.DataFrame(scores)

,fit_time,score_time,test_neg_mean_absolute_error,test_neg_mean_squared_error,test_max_error
0,0.002298,0.000875,-2.312430,-9.325530,-6.449885
1,0.000796,0.000457,-1.746534,-4.944962,-5.589261
2,0.000453,0.000417,-2.562117,-11.396652,-10.339140
3,0.000439,0.000420,-2.018732,-7.024211,-6.619504
4,0.000463,0.000400,-2.279519,-8.385627,-7.755785


In [53]:
pd.DataFrame(scores).mean()

fit_time                        0.000890
score_time                      0.000514
test_neg_mean_absolute_error   -2.183866
test_neg_mean_squared_error    -8.215396
test_max_error                 -7.350715
dtype: float64

**Pas model aan op basis van de resultaten**

In [54]:
model = Ridge(alpha=1)

In [55]:
# SCORING OPTIONS:
# https://scikit-learn.org/stable/modules/model_evaluation.html
scores = cross_validate(model,X_train,y_train,
                         scoring=['neg_mean_absolute_error','neg_mean_squared_error','max_error'],cv=5)

In [56]:
pd.DataFrame(scores).mean()

fit_time                        0.000584
score_time                      0.000530
test_neg_mean_absolute_error   -1.319685
test_neg_mean_squared_error    -3.344839
test_max_error                 -5.161145
dtype: float64

**Finale Evaluatie (Geen parameters meer aanpassen hierna!)**

In [57]:
# Eerst model fitten
model.fit(X_train,y_train)

Ridge(alpha=1)

In [58]:
y_final_test_pred = model.predict(X_test)

In [59]:
mean_squared_error(y_test,y_final_test_pred)

2.3190215794287514

----
----